# Telecom Assignment – Single Notebook
Run top to bottom. All names and paths are defined **once** in the config cell and reused everywhere.

1. Config
2. Create catalog / schema / volumes
3. Create folders (volume + workspace source folder)
4. Copy source files into the landing volume
5. Validate setup
6. Read customer, usage, tower with schemas
7. Write each DataFrame as Parquet / JSON / ORC / CSV
8. Read ORC back and write Parquet
9. Final verification

## 1. Config (defined once)

In [0]:
import os
from pyspark.sql.types import StructType, StructField, StringType, IntegerType

# ---- Unity Catalog objects ----
catalog      = "telecom_catalog_assign"
schema       = "landing_zone"
landing_vol  = "landing_vol"
output_vol   = "output"

# ---- Paths ----
base    = f"/Volumes/{catalog}/{schema}/{landing_vol}"   # input  (landing)
out     = f"/Volumes/{catalog}/{schema}/{output_vol}"    # output
src_dir = "/Workspace/Users/nithish.t1712@gmail.com/DWH-REPO/Databricks/TELECOM_PROJ/source_files"

# ---- Folder layout ----
folders       = ["customer", "usage", "tower"]
tower_regions = ["region1", "region2"]

# source file -> destination folder inside the landing volume
files_to_copy = {
    "customer.csv":           f"{base}/customer",
    "usage.tsv":              f"{base}/usage",
    "tower_logs_region1.txt": f"{base}/tower/region1",
    "tower_logs_region2.txt": f"{base}/tower/region2",
}

# ---- Helpers ----
def check(label, ok):
    print(f"{'PASS' if ok else 'FAIL'}  {label}")
    return ok

def list_names(path):
    """Names of items inside a path, or [] if the path doesn't exist."""
    try:
        return [f.name.rstrip("/") for f in dbutils.fs.ls(path)]
    except Exception:
        return []

def write_all_formats(df, name):
    """Write one DataFrame as parquet, json, orc and csv under `out`."""
    df.write.mode("overwrite").parquet(f"{out}/{name}_parquet")
    df.write.mode("overwrite").json(f"{out}/{name}_json")
    df.write.mode("overwrite").orc(f"{out}/{name}_orc")
    df.write.mode("overwrite").option("header", True).csv(f"{out}/{name}_csv")
    print(f"Written {name} in 4 formats -> {out}")

print("\nAll config is completed")

## 2. Create catalog, schema and volumes

In [0]:
spark.sql(f"CREATE CATALOG IF NOT EXISTS {catalog}")
spark.sql(f"CREATE SCHEMA IF NOT EXISTS {catalog}.{schema}")
spark.sql(f"CREATE VOLUME IF NOT EXISTS {catalog}.{schema}.{landing_vol}")
spark.sql(f"CREATE VOLUME IF NOT EXISTS {catalog}.{schema}.{output_vol}")

display(spark.sql("SHOW CATALOGS"))
display(spark.sql(f"SHOW SCHEMAS IN {catalog}"))
display(spark.sql(f"SHOW VOLUMES IN {catalog}.{schema}"))

## 3. Create folders

In [0]:
# Workspace source folder (regular filesystem, so use os.makedirs, not dbutils.fs.mkdirs)
os.makedirs(src_dir, exist_ok=True)

# Landing volume folders
for folder in folders:
    dbutils.fs.mkdirs(f"{base}/{folder}/")
for region in tower_regions:
    dbutils.fs.mkdirs(f"{base}/tower/{region}/")

display(dbutils.fs.ls(base))
display(dbutils.fs.ls(f"{base}/tower/"))

## 4. Copy source files into the landing volume
Upload the 4 source files into the workspace `source_files` folder first.

In [0]:
missing = [f for f in files_to_copy if not os.path.isfile(f"{src_dir}/{f}")]
if missing:
    raise FileNotFoundError(f"Upload these files to {src_dir} first: {missing}")

for file_name, dest_folder in files_to_copy.items():
    with open(f"{src_dir}/{file_name}", "r", encoding="utf-8") as f:
        content = f.read()
    dbutils.fs.put(f"{dest_folder}/{file_name}", content, overwrite=True)
    print(f"Copied {file_name} -> {dest_folder}/")

## 5. Validate setup

In [0]:
results = []

# catalog, schema, volumes
results.append(check(f"Catalog {catalog}",
    spark.sql(f"SHOW CATALOGS LIKE '{catalog}'").count() == 1))
results.append(check(f"Schema {catalog}.{schema}",
    spark.sql(f"SHOW SCHEMAS IN {catalog} LIKE '{schema}'").count() == 1))
for vol in (landing_vol, output_vol):
    results.append(check(f"Volume {catalog}.{schema}.{vol}",
        spark.sql(f"SHOW VOLUMES IN {catalog}.{schema} LIKE '{vol}'").count() == 1))

# folders in the landing volume
existing = list_names(base)
for folder in folders:
    results.append(check(f"Volume folder {base}/{folder}/", folder in existing))

tower_existing = list_names(f"{base}/tower")
for region in tower_regions:
    results.append(check(f"Region folder {base}/tower/{region}/", region in tower_existing))

# source folder + source files
results.append(check(f"Source folder {src_dir}", os.path.isdir(src_dir)))
for file_name in files_to_copy:
    results.append(check(f"Source file {file_name}", os.path.isfile(f"{src_dir}/{file_name}")))

# files landed in the volume
for file_name, dest_folder in files_to_copy.items():
    results.append(check(f"Landed {dest_folder}/{file_name}", file_name in list_names(dest_folder)))

print("\nAll checks passed" if all(results) else "\nSome checks failed")

## 6. Read the data with schemas

In [0]:
cust_schema = StructType([
    StructField("CustId",   IntegerType(), True),
    StructField("FullName", StringType(),  False),
    StructField("Age",      IntegerType(), False),
    StructField("City",     StringType(),  False),
    StructField("PlanType", StringType(),  False),
])
usage_schema = StructType([
    StructField("CustId",     IntegerType(), True),
    StructField("Usage_mins", IntegerType(), False),
    StructField("Data_MB",    IntegerType(), False),
    StructField("SMS_Count",  IntegerType(), False),
])
tower_schema = StructType([
    StructField("EventId",   StringType(),  False),
    StructField("CustId",    IntegerType(), True),
    StructField("TowerId",   StringType(),  False),
    StructField("Signal",    IntegerType(), False),
    StructField("Timestamp", StringType(),  False),
])

# customer: comma separated, no header
cust_df = (spark.read
    .options(header=False, sep=",", pathGlobFilter="*.csv")
    .schema(cust_schema)
    .csv(f"{base}/customer"))

# usage: tab separated, with header
usage_df = (spark.read
    .options(header=True, sep="\t", pathGlobFilter="*.tsv")
    .schema(usage_schema)
    .csv(f"{base}/usage"))

# tower: pipe separated, with header, region1 + region2 together
tower_df = (spark.read
    .options(header=True, sep="|", pathGlobFilter="*.txt", recursiveFileLookup=True)
    .schema(tower_schema)
    .csv(f"{base}/tower"))

for name, df in [("customer", cust_df), ("usage", usage_df), ("tower", tower_df)]:
    print(f"--- {name}: {df.count()} rows")
    df.printSchema()
    df.show(3)

# sanity check: a header row read as data shows up as a null CustId
print("Null CustId -> customer:", cust_df.filter("CustId IS NULL").count(),
      "| usage:", usage_df.filter("CustId IS NULL").count(),
      "| tower:", tower_df.filter("CustId IS NULL").count())

## 7. Write each DataFrame as Parquet / JSON / ORC / CSV

In [0]:
write_all_formats(cust_df,  "customer")
write_all_formats(usage_df, "usage")
write_all_formats(tower_df, "tower")

## 8. Read ORC back and write Parquet

In [0]:
parquet_out = f"{out}/output_parquet"

for name in ["customer", "usage", "tower"]:
    df_orc = spark.read.orc(f"{out}/{name}_orc")
    print(f"{name}: {df_orc.count()} rows from ORC")
    df_orc.write.mode("overwrite").parquet(f"{parquet_out}/{name}_parquet")

display(dbutils.fs.ls(parquet_out))

## 9. Final verification

In [0]:
for name in ["customer", "usage", "tower"]:
    print(f"--- {name}")
    spark.read.parquet(f"{parquet_out}/{name}_parquet").show(5)

display(dbutils.fs.ls(out))



In [0]:
readers = {
    "parquet": spark.read.parquet,
    "json":    spark.read.json,
    "orc":     spark.read.orc,
    "csv":     lambda p: spark.read.option("header", True).csv(p),
}

rows = []

def count_outputs(path):
    for item in dbutils.fs.ls(path):
        if not item.isDir():
            continue
        name = item.name.rstrip("/")
        fmt = name.rsplit("_", 1)[-1]
        if name == "output_parquet":        # container folder from step 8, look inside it
            count_outputs(item.path)
        elif fmt in readers:
            label = item.path.replace(f"dbfs:{out}/", "").rstrip("/")
            rows.append((label, fmt, readers[fmt](item.path).count()))

count_outputs(out)

display(
    spark.createDataFrame(rows, "folder STRING, format STRING, record_count LONG")
         .orderBy("format")
)